# 自定义中间件-Wrap-style hooks



In [2]:
from itertools import count

from langchain.agents.middleware.types import ResponseT
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

from langgraph.typing import ContextT

# 1、提供大模型
load_dotenv(override=True)

model = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",
    profile={"max_input_tokens": 128_000},
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    # temperature=1.5,
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    extra_body={"enable_thinking": False},
    # max_tokens=10,
)

model_with_zhipuai = init_chat_model(
    model="GLM-4.5-Air",
    model_provider="openai",
    api_key=os.getenv("ZHIPUAI_API_KEY"),
    base_url=os.getenv("ZHIPUAI_BASE_URL"),
    extra_body={"enable_thinking": False}
)

## 1、wrap_model_call的使用

### 1、基于装饰器的实现

In [3]:
from typing import Callable
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse, ExtendedModelResponse


@wrap_model_call
def wrap_model_call_middleware(
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse | None:

    request.messages[-1].content += "----> wrap_model_call_before <----"

    # 模型的调用
    response = handler(request)

    response.result[0].content += "----> wrap_model_call_after <----"

    return response

In [6]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, AIMessage

agent = create_agent(
    model=model,
    middleware=[
        wrap_model_call_middleware,
    ]
)

response = agent.invoke({
    "messages" : [
        HumanMessage("你好！")
    ]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好！----> wrap_model_call_before <----
================================== Ai Message ==================================

你好！有什么我可以帮你的吗？----> wrap_model_call_after <----


### 2、基于类的实现

In [8]:

from langchain.agents.middleware import AgentMiddleware


class WrapModelCallMiddleware(AgentMiddleware):
    def wrap_model_call(self,
                        request: ModelRequest,
                        handler: Callable[[ModelRequest], ModelResponse],
                        ) -> ModelResponse | None:

        request.messages[-1].content += "----> wrap_model_call_before <----"

        # 模型的调用
        response = handler(request)

        response.result[0].content += "----> wrap_model_call_after <----"

        return response

from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, AIMessage

agent = create_agent(
    model=model,
    middleware=[
        WrapModelCallMiddleware(),
    ]
)

response = agent.invoke({
    "messages" : [
        HumanMessage("你好！")
    ]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好！----> wrap_model_call_before <----
================================== Ai Message ==================================

你好！有什么我可以帮你的吗？----> wrap_model_call_after <----


### 场景1：重试逻辑

In [10]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable
import time

@wrap_model_call
def retry_model(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """自动重试失败的模型调用"""

    max_retries = 3

    for attempt in range(max_retries):
        try:
            print(f"✅ 尝试调用模型（第 {attempt + 1}/{max_retries} 次）")
            return handler(request)
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"❌ 所有重试失败：{e}")
                raise

            # 指数退避
            wait_time = 2 ** attempt

            print(f"⚠ 调用失败：{e}，{wait_time} 秒后重试")
            time.sleep(wait_time)

### 场景2：响应缓存

In [ ]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable
import hashlib
import json

class ModelCache:
    """模型响应缓存"""
    def __init__(self):
        self.cache = {}

    def create_hook(self):
        @wrap_model_call
        def cache_model(
            request: ModelRequest,
            handler: Callable[[ModelRequest], ModelResponse]
        ) -> ModelResponse:
            # 生成缓存键
            cache_key = hashlib.md5(
                json.dumps({
                    "messages": [str(m) for m in request.messages],
                    "system": str(request.system_message)
                }).encode()
            ).hexdigest()
            # 检查缓存
            if cache_key in self.cache:
                print("✅ 缓存命中！")
                return self.cache[cache_key]
            # 调用模型
            print("⏳ 缓存未命中，调用模型")
            response = handler(request)
            # 存入缓存
            self.cache[cache_key] = response
            return response
        return cache_model

# 使用
cache = ModelCache()
agent = create_agent(
    model=model,
    middleware=[cache.create_hook()]
)

### 场景3：修改系统提示


In [ ]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain_core.messages import SystemMessage
from typing import Callable
from datetime import datetime

@wrap_model_call
def add_context(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """动态添加上下文信息到系统提示"""

    # 获取当前时间
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # 构建新的系统消息
    original_content = request.system_message.content if request.system_message else ""
    new_content = f"""{original_content}

         当前时间：{current_time}
         用户位置：中国
         语言偏好：中文
         """

    # 创建新的系统消息
    new_system_message = SystemMessage(content=new_content)

    # 使用 override 方法修改请求（不可变，返回新请求对象）
    modified_request = request.override(system_message=new_system_message)

    return handler(modified_request)

## 2、wrap_tool_call的使用

### 2.1 基于装饰器的实现